# Agent Network Protocol (ANP) | Agent Protocols

In [1]:
# ANP: Decentralized Identity + Mutual Trust Verification (no central registry)
# Production ANP uses Ed25519/X25519 keys. This demo uses HMAC for simplified but meaningful signing.
import hashlib, hmac, os
from dataclasses import dataclass, field
from typing import Dict, List, Optional

In [2]:
@dataclass
class DIDDocument:
    """W3C DID Document with signing capabilities and advertised skills."""
    did: str
    name: str
    capabilities: List[str]
    _secret: bytes = field(default_factory=lambda: os.urandom(32), repr=False)
    verification_key: str = ""  # public portion (hash of secret, shared in DID doc)

    def __post_init__(self):
        self.verification_key = hashlib.sha256(self._secret).hexdigest()

    def sign(self, message: str) -> str:
        """Sign a message with this agent's secret key (HMAC-SHA256)."""
        return hmac.new(self._secret, message.encode(), hashlib.sha256).hexdigest()

    def verify(self, message: str, signature: str) -> bool:
        """Verify a signature matches this agent's key."""
        expected = hmac.new(self._secret, message.encode(), hashlib.sha256).hexdigest()
        return hmac.compare_digest(expected, signature)

@dataclass
class SignedMessage:
    sender_did: str
    content: str
    signature: str  # HMAC signature over content

class ANPAgent:
    """An agent that discovers peers, verifies identity, and communicates securely."""
    def __init__(self, did_doc: DIDDocument):
        self.did_doc = did_doc
        self.trusted_peers: Dict[str, DIDDocument] = {}  # verified peers

    def publish_did(self) -> DIDDocument:
        """Publish DID document (in real ANP, this goes to a decentralized resolver)."""
        return self.did_doc

    def verify_and_trust(self, peer_doc: DIDDocument) -> bool:
        """Verify a peer's DID by checking their self-signature (no central authority)."""
        challenge = f"trust-challenge-{self.did_doc.did}-{peer_doc.did}"
        sig = peer_doc.sign(challenge)
        if peer_doc.verify(challenge, sig):
            self.trusted_peers[peer_doc.did] = peer_doc
            return True
        return False

    def send(self, target_did: str, content: str) -> SignedMessage:
        """Send a signed message to a trusted peer."""
        assert target_did in self.trusted_peers, f"Untrusted peer: {target_did}"
        sig = self.did_doc.sign(content)
        return SignedMessage(sender_did=self.did_doc.did, content=content, signature=sig)

    def receive(self, msg: SignedMessage) -> Optional[str]:
        """Receive and verify a message -- reject if signature invalid."""
        peer_doc = self.trusted_peers.get(msg.sender_did)
        if not peer_doc or not peer_doc.verify(msg.content, msg.signature):
            print(f"  REJECTED: invalid signature from {msg.sender_did}")
            return None
        return msg.content

In [3]:
# --- Create two agents with independent DIDs (no shared registry) ---
agent_a = ANPAgent(DIDDocument(did="did:web:acme.com:analyst", name="Acme Analyst",
                               capabilities=["data_analysis", "forecasting"]))
agent_b = ANPAgent(DIDDocument(did="did:web:beta.io:reporter", name="Beta Reporter",
                               capabilities=["writing", "summarization"]))

# Step 1: Discover -- agents exchange DID documents directly (peer-to-peer)
doc_a = agent_a.publish_did()
doc_b = agent_b.publish_did()
print(f"Agent A publishes: {doc_a.did} capabilities={doc_a.capabilities}")
print(f"Agent B publishes: {doc_b.did} capabilities={doc_b.capabilities}")

# Step 2: Mutual verification -- each agent verifies the other (no central authority)
ok_ab = agent_a.verify_and_trust(doc_b)
ok_ba = agent_b.verify_and_trust(doc_a)
print(f"\nA trusts B: {ok_ab} | B trusts A: {ok_ba}")
print("Trust established via mutual DID verification")

# Step 3: Signed communication -- messages are cryptographically signed
msg1 = agent_a.send(doc_b.did, "Analyze Q3 revenue data: $2.4M, 12.5% growth")
received = agent_b.receive(msg1)
print(f"\nA -> B (signed): {received}")

msg2 = agent_b.send(doc_a.did, "Report complete. Cloud segment drives 60% of revenue.")
received2 = agent_a.receive(msg2)
print(f"B -> A (signed): {received2}")

# Step 4: Show that tampered messages are rejected
tampered = SignedMessage(sender_did=doc_a.did, content="TAMPERED DATA", signature="bad_sig")
result = agent_b.receive(tampered)
print(f"\nTampered message accepted? {result is not None}  (rejected as expected)")

Agent A publishes: did:web:acme.com:analyst capabilities=['data_analysis', 'forecasting']
Agent B publishes: did:web:beta.io:reporter capabilities=['writing', 'summarization']

A trusts B: True | B trusts A: True
Trust established via mutual DID verification

A -> B (signed): Analyze Q3 revenue data: $2.4M, 12.5% growth
B -> A (signed): Report complete. Cloud segment drives 60% of revenue.
  REJECTED: invalid signature from did:web:acme.com:analyst

Tampered message accepted? False  (rejected as expected)
